# Step 8 — Monitoring

Track gap rate and reconstruction quality across the dataset.
Flag any unusual patterns that might indicate data or model drift.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

BASE_DIR  = Path("../noel_part")
CLEAN_DIR = BASE_DIR / "cleaned_data_final"
OUT_DIR   = Path("../outputs")
OUT_DIR.mkdir(exist_ok=True)

eval_path = OUT_DIR / "evaluation_results.csv"
if eval_path.exists():
    eval_df = pd.read_csv(eval_path)
    print(f"Loaded evaluation results: {len(eval_df)} flights")
else:
    print("Run 05_evaluate.ipynb first to generate evaluation_results.csv")
    eval_df = pd.DataFrame()

## 8a. Gap rate monitoring

In [ ]:
flights = [f for s in sorted(CLEAN_DIR.iterdir()) if s.is_dir()
           for f in sorted(s.iterdir()) if f.is_dir()]

gaps = []
for fd in flights:
    try:
        b = pd.read_parquet(fd / "adsb_before.parquet")
        a = pd.read_parquet(fd / "adsb_after.parquet")
        gap = (a["timestamp"].min() - b["timestamp"].max()).total_seconds() / 60
        step = fd.parent.name
        gaps.append({"flight": fd.name, "step": step, "gap_min": round(gap,1)})
    except: continue

gap_df = pd.DataFrame(gaps)
print(f"Total flights    : {len(gap_df)}")
print(f"Mean gap         : {gap_df['gap_min'].mean():.1f} min")
print(f"Flights > 4h gap : {(gap_df['gap_min'] > 240).sum()}")
print(f"Flights > 5h gap : {(gap_df['gap_min'] > 300).sum()}")

# Alert if any step has unusually few flights
step_counts = gap_df.groupby("step").size()
mean_count  = step_counts.mean()
alerts = step_counts[step_counts < mean_count * 0.5]
if len(alerts):
    print(f"\n⚠ Steps with unusually few flights:")
    print(alerts)
else:
    print("\n✓ Gap rate looks stable across all steps")

## 8b. Reconstruction quality monitoring

In [ ]:
if len(eval_df) > 0:
    print("Reconstruction quality summary:")
    print(f"  Baseline mean error : {eval_df['baseline_mean_m'].mean()/1000:.1f} km")
    if "kalman_mean_m" in eval_df:
        print(f"  Kalman mean error   : {eval_df['kalman_mean_m'].mean()/1000:.1f} km")
    if "bigru_mean_m" in eval_df:
        print(f"  BiGRU mean error    : {eval_df['bigru_mean_m'].mean()/1000:.1f} km")

    # Alert if BiGRU is worse than baseline
    if "bigru_mean_m" in eval_df and "baseline_mean_m" in eval_df:
        bigru_worse = (eval_df["bigru_mean_m"] > eval_df["baseline_mean_m"]).mean() * 100
        if bigru_worse > 40:
            print(f"\n⚠ BiGRU worse than baseline on {bigru_worse:.0f}% of flights — consider retraining")
        else:
            print(f"\n✓ BiGRU beats baseline on {100-bigru_worse:.0f}% of flights")
else:
    print("No evaluation data — run 05_evaluate.ipynb first")

## 8c. Quality report

In [ ]:
report = {
    "total_flights":   len(gap_df),
    "mean_gap_min":    round(gap_df["gap_min"].mean(), 1),
    "max_gap_min":     round(gap_df["gap_min"].max(), 1),
    "flights_over_4h": int((gap_df["gap_min"] > 240).sum()),
}
if len(eval_df) > 0:
    report["baseline_mean_km"] = round(eval_df["baseline_mean_m"].mean()/1000, 1)
    if "kalman_mean_m" in eval_df:
        report["kalman_mean_km"] = round(eval_df["kalman_mean_m"].mean()/1000, 1)
    if "bigru_mean_m" in eval_df:
        report["bigru_mean_km"] = round(eval_df["bigru_mean_m"].mean()/1000, 1)

import json
report_path = OUT_DIR / "monitoring_report.json"
report_path.write_text(json.dumps(report, indent=2))
print("Monitoring Report:")
print(json.dumps(report, indent=2))
print(f"\nSaved → {report_path}")